# 01 EDA
Run `scripts/build_dataset.py` first.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from ytclickbait.config import load_config

cfg = load_config()
videos = pd.read_parquet(cfg.file("videos"))
split = pd.read_parquet(cfg.file("split"))
sns.set_theme(style="whitegrid")

In [ ]:
print(f"{len(videos):,} videos, {videos['channel_id'].nunique():,} channels")
print(
    "Trending dates:",
    videos["first_trending_date"].min().date(),
    "to",
    videos["trending_date"].max().date(),
)
videos["countries"].explode().value_counts()

## Title changes
How often does a video trend under more than one title?

In [ ]:
print(f"{videos['title_changed'].mean():.1%} of videos had more than one title")
videos.loc[videos["title_changed"], ["first_title", "last_title", "n_distinct_titles"]].head(20)

## Targets

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.histplot(videos["log_views"], ax=axes[0])
sns.histplot(videos["log_likes_per_view"].dropna(), ax=axes[1])
sns.histplot(videos["log_comments_per_view"].dropna(), ax=axes[2])
plt.tight_layout()

## By category

In [ ]:
order = videos["category_name"].value_counts().index
plt.figure(figsize=(10, 6))
sns.boxplot(data=videos, y="category_name", x="log_views", order=order)
videos["category_name"].value_counts()

## Video age vs views
Age is a main control: older snapshots have had more time to collect views.

In [ ]:
sample = videos.sample(min(5000, len(videos)), random_state=0)
sns.scatterplot(data=sample, x="log_age_days", y="log_views", alpha=0.2)

## Channel concentration
How much of the data comes from a few big channels? This is why channel controls matter.

In [ ]:
per_channel = videos["channel_id"].value_counts()
print(per_channel.describe())
print("Top 20 channels' share of videos:", f"{per_channel.head(20).sum() / len(videos):.1%}")
split[["split_group", "split_time"]].apply(pd.Series.value_counts)